In [ ]:
from pathlib import Path

# We use a slightly modified version of the biotrove_process code from the original biotrove repository
from smartrodent.biotrove_process import (
    GenImgTxtPair,
    GenShuffledChunks,
    GetImages,
    MetadataProcessor,
    load_config,
)
from smartrodent.dataprocessing import ImageFilter

In [ ]:
basepath = Path("./").resolve()
basepath

In [ ]:
config_path = basepath / "configs" / "config_central_europe.json"

In [ ]:
datapath = basepath / "datasets" / "BioTrove" / "BioTrove"

# build the image dataset now

In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)

In [ ]:
%load_ext autoreload
%autoreload 2

params = config.get('metadata_processor_info', {})
mp = MetadataProcessor(**params)
mp.process_all_files()

# Step 2: generate shuffled chunks of metadata

filter parquet files first.
TODO: put this into the processing package, after making sure it's needed. 

In [ ]:
from pathlib import Path
import polars as pl

src = Path(config["metadata_processor_info"]["source_folder"])
filtered_dir = Path(config["filter_dir"])
categories = config["metadata_processor_info"]["categories"]
category_type = config["metadata_processor_info"]["category_type"]
filtered_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(src.glob("*.parquet")):
    df = (
        pl.scan_parquet(path)
        .filter(pl.col(category_type).is_in(categories))
        .collect()
    )

    if df.height:
        df.write_parquet(filtered_dir / path.name)

then build chunks

In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)
params = config.get('metadata_filter_and_shuffle_info', {})
params["directory"] = str(filtered_dir)
params

In [ ]:
%load_ext autoreload
%autoreload 2
gen_shuffled_chunks = GenShuffledChunks(**params)
gen_shuffled_chunks.process_files()

# Step 3: Download images


In [ ]:
%load_ext autoreload
%autoreload 2

config = load_config(config_path)

In [ ]:
params = config.get('image_download_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2

gi = GetImages(**params)
await gi.download_images()

# Step 4: Generate text pairs and create tar files (optional)

In [ ]:
%load_ext autoreload
%autoreload 2
config = load_config(config_path)
params = config.get('img_text_gen_info', {})
params

In [ ]:
%load_ext autoreload
%autoreload 2
textgen = GenImgTxtPair(**params)
textgen.create_image_text_pairs()

## Step 5: Generate classifier dataset

In [ ]:
import shutil

In [ ]:
img_dir = config["img_dir"]
Path(img_dir).mkdir(exist_ok=True, parents = True)

In [ ]:
params = config.get('img_text_gen_info', {})

for chunk in Path(params["img_folder"]).iterdir():
    if chunk.is_dir():
        jpgs = [c for c in Path(chunk).iterdir() if c.suffix == ".jpg"]
        scnames = {c.stem.split(".")[0]: c for c in Path(chunk).iterdir() if 'scientific_name' in str(c)}

        for jpg in jpgs:
            name = jpg.stem

            # find scientific name file
            scientific_name_file = scnames[name]

            with open(scientific_name_file, mode = "r") as scf:
                scientific_name = scf.read()[0:-1]

            species_img_path = (Path(img_dir) / scientific_name)
            species_img_path.mkdir(parents=True, exist_ok=True)

            print(scientific_name, ", ", jpg.name )
            # copy the jpg to the species_img_path
            shutil.copy2(
                jpg,
                str(species_img_path / Path(jpg).name)
            )



# Filtering using CLIP by OpenAI 

This is mostly based on https://colab.research.google.com/github/openai/clip/blob/master/notebooks/Interacting_with_CLIP.ipynb#scrollTo=0BpdJkdBssk9

In [ ]:
import clip
import numpy as np
import torch


check that you have cuda and stuff

In [ ]:
clip.available_models()

define prompts for clip. goal is to filter out any images of dead animals (e.g., traps, roadkill) b/c in citizen scientist datasets that's a common source of images for rodents in particular, and especially for rats an mice

In [ ]:
healthy_prompt = "a live healthy looking wild animal"
dead_prompt = "a dead animal, roadkill, a skull or bones"
not_prompt = "not an animal at all"
rodent_prompt = " a mouse, rat or other rodent"

primary_prompts = [ not_prompt, rodent_prompt]
healthy_or_dead_prompt = [healthy_prompt, dead_prompt]

## Feed in example images

try stuff out a bit first. 

In [ ]:

from pathlib import Path
import random


In [ ]:
imgfilter = ImageFilter(model = 'RN50x16', prompts=primary_prompts, id_tol = 0.02)

In [ ]:
path = basepath / "datasets" / "biotrove-central-europe" / "imgs" / "Rattus rattus"

In [ ]:
rng_seed = 32456
sample_size = 10
rng = random.Random(rng_seed)
image_paths = rng.sample(list(path.iterdir()), sample_size)

In [ ]:
original_images, images = imgfilter.preprocess_images(image_paths)

In [ ]:
imgfilter.plot_images(original_images)

In [ ]:
similarity = imgfilter.compute_similarity(images)

In [ ]:
decided_idx, undecided_idx, decided, undecided = imgfilter.filter_similarities(
    similarity
)

In [ ]:
decided

In [ ]:
decided.argmax(dim=0)

In [ ]:
decided_idx

In [ ]:
imgfilter.plot_sim_score(decided, [original_images[i] for i in decided_idx], )

In [ ]:
imgfilter.plot_sim_score(undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
rodent_images = [images[i] for i in decided_idx]
rodent_original_images = [original_images[i] for i in decided_idx]

In [ ]:
rodent_filter = ImageFilter('RN50x16', prompts = healthy_or_dead_prompt, id_tol =0.02)

In [ ]:
# original_images, images = rodent_filter.preprocess_images(rodent_images)
rodent_similarity = rodent_filter.compute_similarity(rodent_images)


In [ ]:
rodent_decided_idx, rodent_undecided_idx, rodent_decided, rodent_undecided = rodent_filter.filter_similarities(
    rodent_similarity
)

In [ ]:
rodent_filter.plot_sim_score(rodent_decided, [original_images[i] for i in rodent_decided_idx], )

In [ ]:
rodent_decided.argmax(dim=0)

In [ ]:
rodent_filter.plot_sim_score(rodent_undecided, [original_images[i] for i in undecided_idx], )

In [ ]:
labeled_data = rodent_filter.decisions(rodent_original_images, rodent_decided)

In [ ]:
labeled_data

In [ ]:
imgfilter.plot_images(labeled_data['a dead animal, roadkill, a skull or bones'])

In [ ]:
imgfilter.plot_images(labeled_data['a live healthy looking wild animal'])

## Sequential CLIP filtering for all Central Europe species

This cell walks every species folder under `datasets/biotrove-central-europe/imgs`, first keeps images that CLIP labels as rodent-like, then keeps only the subset also labeled as live/healthy. Kept images are copied into `filtered/` and all rejected, dead, or uncertain images are copied into `filtered_dead_or_unsure/`, preserving the species-folder label structure.

WARNING: LLM generated, check if it works as intended

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import shutil

import torch
from tqdm.auto import tqdm

from smartrodent.dataprocessing import ImageFilter

# Source directory: one subdirectory per species; each subdirectory name is the label.
imgs_root = Path(
    "datasets/biotrove-central-europe/imgs"
)
filtered_root = imgs_root.parent / "filtered"
rejected_root = imgs_root.parent / "filtered_dead_or_unsure"

# Leave this False to avoid deleting previous results accidentally. Set to True
# when rebuilding the filtered directories from scratch.
clear_existing_outputs = False

if clear_existing_outputs:
    shutil.rmtree(filtered_root, ignore_errors=True)
    shutil.rmtree(rejected_root, ignore_errors=True)

filtered_root.mkdir(parents=True, exist_ok=True)
rejected_root.mkdir(parents=True, exist_ok=True)

image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# First pass: animal/rodent-like versus not-an-animal.
rodent_prompt = "a mouse, rat or other rodent"
not_prompt = "not an animal at all"

# Second pass: usable live-animal images versus dead/specimen/uncertain images.
healthy_prompt = "a live healthy looking wild animal"
dead_or_unsure_prompt = "a dead animal, roadkill, a skull or bones"

primary_filter = ImageFilter(
    model="RN50x16",
    prompts=[not_prompt, rodent_prompt],
    id_tol=0.02,
)
health_filter = ImageFilter(
    model="RN50x16",
    prompts=[healthy_prompt, dead_or_unsure_prompt],
    id_tol=0.02,
)


def copy_with_structure(src: Path, dst_root: Path) -> Path:
    """Copy one image to dst_root while preserving its path below imgs_root."""
    relative_path = src.relative_to(imgs_root)
    dst = dst_root / relative_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst


def decided_label(image_filter: ImageFilter, preprocessed_images: list) -> tuple[str | None, torch.Tensor]:
    """Return the winning prompt only when CLIP made a non-ambiguous decision."""
    similarity = image_filter.compute_similarity(preprocessed_images)
    decided_idx, _, decided, _ = image_filter.filter_similarities(similarity)
    if len(decided_idx) != 1:
        return None, similarity
    return image_filter.prompts[int(decided.argmax(dim=0).item())], similarity


species_dirs = sorted(p for p in imgs_root.iterdir() if p.is_dir())
image_paths = [
    image_path
    for species_dir in species_dirs
    for image_path in sorted(species_dir.rglob("*"))
    if image_path.is_file() and image_path.suffix.lower() in image_suffixes
]

summary = Counter()
by_species = defaultdict(Counter)
errors = []

for image_path in tqdm(image_paths, desc="CLIP filtering images"):
    species = image_path.relative_to(imgs_root).parts[0]
    try:
        _, preprocessed = primary_filter.preprocess_images([image_path])
        primary_label, _ = decided_label(primary_filter, preprocessed)

        if primary_label != rodent_prompt:
            copy_with_structure(image_path, rejected_root)
            reason = "not_rodent_or_primary_unsure"
        else:
            health_label, _ = decided_label(health_filter, preprocessed)
            if health_label == healthy_prompt:
                copy_with_structure(image_path, filtered_root)
                reason = "kept_live_rodent"
            else:
                copy_with_structure(image_path, rejected_root)
                reason = "dead_or_health_unsure"

        summary[reason] += 1
        by_species[species][reason] += 1
    except Exception as exc:
        copy_with_structure(image_path, rejected_root)
        summary["error"] += 1
        by_species[species]["error"] += 1
        errors.append((str(image_path), repr(exc)))

print("Input:", imgs_root)
print("Kept live rodent images:", filtered_root)
print("Rejected/dead/unsure images:", rejected_root)
print("Total images:", len(image_paths))
print("Summary:", dict(summary))
print("By species:")
for species, counts in sorted(by_species.items()):
    print(species, dict(counts))

if errors:
    print(f"Encountered {len(errors)} errors; first 10:")
    for path, error in errors[:10]:
        print(path, error)

## Visualize datasets created

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
data_dir = "../datasets"
dataset = "biotrove-central-europe"
img_dir = "filtered"
unused_dir = "filtered_dead_or_unsure"

In [ ]:
used_path = Path(data_dir).resolve() / dataset / img_dir

In [ ]:
unused_path = Path(data_dir).resolve() / dataset / unused_dir

In [ ]:
used_data = dict()
for p in used_path.iterdir():
    used_data[p.name] = [len(list(p.iterdir())),]
used_data

In [ ]:
unused_data = dict()
for p in unused_path.iterdir():
    unused_data[p.name] = [len(list(p.iterdir())),]
unused_data

In [ ]:
used_data = pd.DataFrame.from_dict(used_data)
used_data

In [ ]:
unused_data = pd.DataFrame.from_dict(unused_data)
unused_data

## Generate dataset from filtered images and speciesnet results 

In [ ]:
from smartrodent.dataprocessing import YoloDatasetCreatorFromSpeciesnet

In [ ]:
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1

In [ ]:
dataset_output_path = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/yolo_training_dataset"
path_to_data = "/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered"
path_to_labels = "/home/hmack/Development/rodent_experiments/runs/detect02/speciesnet/central-europe"
class_names = [
    "Rattus norvegicus",
    "Rattus rattus",
    "Mus musculus",
    "Myodes glareolus",
    "Apodemus agrarius",
    "Apodemus flavicollis",
    "Apodemus sylvaticus",
    "Microtus arvalis",
    "Microtus agrestis",
    "Crocidura leucodon",
]
image_directory = "boxed"

In [ ]:
dataset_creator = YoloDatasetCreatorFromSpeciesnet(
    path_to_data = path_to_data,
    path_to_labels = path_to_labels,
    dataset_output_path = dataset_output_path,
    confidence_threshold = 0.1,
    labels_to_filter = ['animal', 'rodent'],
    class_names = class_names,
    image_directory = image_directory
)

In [ ]:
dataset = dataset_creator()